# Baseline CNN on CIFAR-100 — Curated Experiment Suite

Standalone Colab-friendly notebook. Trains the baseline CNN on six curated
CIFAR-100 tasks and prints a compact comparison table.

**Experiments (in order):**
1. `food_containers` — coarse binary, strong object baseline
2. `aquatic_mammals` — coarse binary, harder animal baseline
3. `skyscraper` — fine binary, strongest single-class result
4. `orange` — fine binary, plain baseline
5. `orange_aug_light` — fine binary, light-augmentation ablation vs. orange
6. `coarse_multiclass` — 20-way coarse classification, motivates transfer learning

**Convention:** Does NOT import from local `data/`, `models/`, `training/`, or
`evaluation/`. Everything is inlined for Colab portability.

## 1. Setup

Install missing dependencies via `subprocess` (avoiding `%pip` magic so static type checkers do not flag the notebook). Then import, seed NumPy / TensorFlow / Python `random`, and define the global config dict that drives the run.


In [ ]:
# Install dependencies on demand for a fresh Colab environment. We gate every install
# on a successful import to avoid re-resolving an already-working package set, which
# in particular protects the typing_extensions version that the local kernel needs.
import subprocess
import sys


def _ensure(module_name: str, pip_name: str | None = None) -> None:
    try:
        __import__(module_name)
    except ImportError:
        target = pip_name or module_name
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", target]
        )


_ensure("datasets")
_ensure("sklearn", "scikit-learn")
_ensure("yaml", "pyyaml")
_ensure("matplotlib")


In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")  # suppress Metal INFO logs

import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import yaml
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("tensorflow:", tf.__version__)
print("numpy:     ", np.__version__)

In [ ]:
# Flip to False for full CIFAR-100 runs with curated epoch counts.
FAST_DEV_RUN = True

DEFAULT_EPOCHS = {
    "food_containers":   20,
    "aquatic_mammals":   30,
    "skyscraper":        80,
    "orange":            40,
    "orange_aug_light":  40,
    "coarse_multiclass": 30,
}

FAST_EPOCHS = {k: 2 for k in DEFAULT_EPOCHS}
FAST_SUBSET = 1024

OUTPUT_DIR = Path("tmp/baseline_cnn_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("FAST_DEV_RUN:", FAST_DEV_RUN)
print("output dir:", OUTPUT_DIR.resolve())

## 2. Load CIFAR-100 from Hugging Face

We pull both splits and convert the PIL images into `(N, 32, 32, 3) uint8` NumPy arrays. The first load downloads ~170 MB and caches under `~/.cache/huggingface/`.


In [ ]:
ds = load_dataset("uoft-cs/cifar100")
train_split = ds["train"]
test_split = ds["test"]

fine_names = train_split.features["fine_label"].names
coarse_names = train_split.features["coarse_label"].names


def split_to_arrays(split):
    """Convert a HF split into (images uint8, fine int64, coarse int64) arrays."""
    images = np.stack(
        [np.asarray(row["img"].convert("RGB"), dtype=np.uint8) for row in split]
    )
    fine = np.asarray(split["fine_label"], dtype=np.int64)
    coarse = np.asarray(split["coarse_label"], dtype=np.int64)
    return images, fine, coarse


train_images, train_fine, train_coarse = split_to_arrays(train_split)
test_images, test_fine, test_coarse = split_to_arrays(test_split)

print("train images:", train_images.shape, train_images.dtype)
print("test  images:", test_images.shape, test_images.dtype)
print("coarse range:", int(train_coarse.min()), int(train_coarse.max()))


In [ ]:
# Label-map helpers — keep these explicit so every experiment references
# names, not magic integers.
FINE_ID   = {name: i for i, name in enumerate(fine_names)}
COARSE_ID = {name: i for i, name in enumerate(coarse_names)}

print("fine classes (first 10):",  list(FINE_ID.items())[:10])
print("coarse classes:", list(COARSE_ID.items()))

## 3. Build the Binary Task

Reusable binary and multiclass task builders. The runner loop selects between them per experiment.

In [ ]:
def make_binary_labels(labels: np.ndarray, positive_ids) -> np.ndarray:
    """Return int64 {0, 1} labels; members of positive_ids become 1."""
    positive_set = {int(i) for i in positive_ids}
    if not positive_set:
        return np.zeros_like(labels, dtype=np.int64)
    return np.isin(labels, list(positive_set)).astype(np.int64)


def label_maps():
    """Build label-name maps from the loaded CIFAR-100 metadata."""
    missing = [name for name in ("fine_names", "coarse_names") if name not in globals()]
    if missing:
        raise RuntimeError(
            "Run the CIFAR-100 loading cell before task-builder/runner cells; "
            f"missing: {', '.join(missing)}"
        )
    return (
        {name: i for i, name in enumerate(fine_names)},
        {name: i for i, name in enumerate(coarse_names)},
    )


def binary_task(label_level: str, positive_names: list[str]):
    """Return (y_train, y_test, task_description_str) for a binary task."""
    fine_id, coarse_id = label_maps()
    if label_level == "coarse":
        pos_ids = [coarse_id[n] for n in positive_names]
        y_tr = make_binary_labels(train_coarse, pos_ids)
        y_te = make_binary_labels(test_coarse,  pos_ids)
    else:
        pos_ids = [fine_id[n] for n in positive_names]
        y_tr = make_binary_labels(train_fine, pos_ids)
        y_te = make_binary_labels(test_fine,  pos_ids)
    desc = f"{label_level}/{'+'.join(positive_names)} vs rest"
    print(f"task: {desc}  train {dict(Counter(y_tr.tolist()))}  test {dict(Counter(y_te.tolist()))}")
    return y_tr, y_te, desc


def multiclass_task():
    """Return (y_train, y_test, description) for 20-way coarse classification."""
    desc = "coarse multiclass (20-way)"
    print(f"task: {desc}  train classes: {len(np.unique(train_coarse))}")
    return train_coarse.astype(np.int64), test_coarse.astype(np.int64), desc

## 4. Stratified Subset for the Smoke Run

Naive head slicing of CIFAR-100 can leave the rare positive class out of the smoke subset. We instead take a deterministic stratified subset (matching `training/train.py::_stratified_subset`).


In [ ]:
def stratified_subset(images, labels, *, subset_size, seed):
    """Deterministic stratified subset over all labels present."""
    if subset_size is None or subset_size >= labels.shape[0]:
        return images, labels
    if subset_size < 2:
        raise ValueError("subset_size must be at least 2 when set")

    rng = np.random.default_rng(seed)
    selected_parts = []
    classes = np.unique(labels)
    for cls in classes:
        cls_idx = np.flatnonzero(labels == cls)
        n_cls = max(1, int(round(subset_size * cls_idx.size / labels.shape[0])))
        n_cls = min(n_cls, cls_idx.size)
        selected_parts.append(rng.choice(cls_idx, size=n_cls, replace=False))

    selected = np.concatenate(selected_parts)
    if selected.shape[0] > subset_size:
        selected = rng.choice(selected, size=subset_size, replace=False)
    elif selected.shape[0] < subset_size:
        remaining = np.setdiff1d(
            np.arange(labels.shape[0]),
            selected,
            assume_unique=False,
        )
        extra = rng.choice(
            remaining,
            size=min(subset_size - selected.shape[0], remaining.shape[0]),
            replace=False,
        )
        selected = np.concatenate([selected, extra])

    rng.shuffle(selected)
    return images[selected], labels[selected]

## 5. Stratified Train/Validation Split

Reimplements `training/splits.py::stratified_train_val_split`: shuffle per-class indices with a seeded RNG, peel off `val_fraction` per class, reshuffle the union so positives and negatives are interleaved.


In [ ]:
def stratified_train_val_split(images, labels, *, val_fraction=0.1, seed=42):
    """Split into train/validation with per-class sampling."""
    rng = np.random.default_rng(seed)
    train_parts, val_parts = [], []
    for cls in np.unique(labels):
        idx = np.flatnonzero(labels == cls)
        rng.shuffle(idx)
        n_val = max(1, int(round(len(idx) * val_fraction)))
        n_val = min(n_val, max(len(idx) - 1, 1))
        val_parts.append(idx[:n_val])
        train_parts.append(idx[n_val:])

    train_idx = np.concatenate(train_parts)
    val_idx = np.concatenate(val_parts)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return images[train_idx], labels[train_idx], images[val_idx], labels[val_idx]

## 6. `tf.data` Pipelines

Mirrors `data.make_pipeline(view="image")`: cast to float32, normalize to `[0, 1]`, shuffle (train only) with a seeded buffer, batch, prefetch.


In [ ]:
def make_pipeline(images, labels, *, batch_size, shuffle, shuffle_buffer=1024, seed=None):
    """Build a tf.data.Dataset that yields (float32 image in [0,1], int64 label)."""
    images_f32 = images.astype(np.float32, copy=False) / 255.0
    labels_i64 = labels.astype(np.int64, copy=False)
    ds = tf.data.Dataset.from_tensor_slices((images_f32, labels_i64))
    if shuffle:
        ds = ds.shuffle(
            buffer_size=shuffle_buffer,
            seed=seed,
            reshuffle_each_iteration=False,
        )
    ds = ds.batch(batch_size, drop_remainder=False)
    return ds.prefetch(tf.data.AUTOTUNE)

## 7. Baseline CNN Model

Inlined from `models/baseline.py`: two conv blocks (32, then 64 filters), max-pool + dropout after each, then a 128-unit dense head with sigmoid output. Returned uncompiled so we can compile it explicitly below.


In [ ]:
def make_augmentation_layer() -> tf.keras.Sequential:
    """Light augmentation — horizontal flip, small translation, small zoom.

    Rotation is intentionally disabled: it is safe for round objects like
    orange but hurts classes like skyscraper, vehicles, and people.
    Attach only for the orange_aug_light experiment.
    """
    return tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomTranslation(0.05, 0.05),
            tf.keras.layers.RandomZoom(0.05),
        ],
        name="augmentation",
    )

In [ ]:
keras  = tf.keras
layers = tf.keras.layers


def build_baseline_cnn(
    input_shape=(32, 32, 3),
    dropout: float = 0.3,
    num_classes: int = 1,
    augmentation=None,
):
    """Two-block CNN. augmentation: optional Keras layer prepended before conv blocks."""
    inputs = keras.Input(shape=input_shape, name="image")
    x = augmentation(inputs) if augmentation is not None else inputs
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D(pool_size=2)(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D(pool_size=2)(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    if num_classes == 1:
        outputs = layers.Dense(1, activation="sigmoid", name="prob")(x)
    else:
        outputs = layers.Dense(num_classes, activation="softmax", name="prob")(x)
    return keras.Model(inputs=inputs, outputs=outputs, name="baseline_cnn")

## 8. Curated Experiment Runner

Evaluation helpers for binary and multiclass tasks, the six curated experiment configs, and
the runner loop. Each experiment appends a metrics dict to `all_results`.

In [ ]:
def compute_balanced_class_weights(labels):
    """sklearn-balanced class weights for binary labels."""
    labels = np.asarray(labels).reshape(-1)
    classes = np.unique(labels)
    if classes.size < 2:
        return {0: 1.0, 1: 1.0}
    w = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    return {int(c): float(v) for c, v in zip(classes, w)}

In [ ]:
def evaluate_binary(model, val_ds, test_ds, y_val, y_test):
    """Evaluate binary model. Selects best threshold on val F1. Returns metrics dict."""
    # Best threshold on validation
    val_prob = model.predict(val_ds, verbose=0).reshape(-1)
    best_thresh, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.91, 0.05):
        f = f1_score(y_val, (val_prob >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_thresh = f, t

    # Test evaluation at best threshold
    test_prob = model.predict(test_ds, verbose=0).reshape(-1)
    y_pred = (test_prob >= best_thresh).astype(np.int64)
    y_true = y_test.astype(np.int64)

    try:
        roc_auc = float(roc_auc_score(y_true, test_prob))
    except ValueError:
        roc_auc = float("nan")
    try:
        pr_auc = float(average_precision_score(y_true, test_prob))
    except ValueError:
        pr_auc = float("nan")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()
    return {
        "threshold": round(float(best_thresh), 2),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc":   roc_auc,
        "pr_auc":    pr_auc,
        "confusion_matrix": cm,
    }

In [ ]:
def top_k_accuracy(y_true, y_prob, k):
    """Fraction of samples where true label is in the top-k predicted classes."""
    top_k = np.argsort(y_prob, axis=1)[:, -k:]
    return float(np.mean([y_true[i] in top_k[i] for i in range(len(y_true))]))


def evaluate_multiclass(model, test_ds, y_test):
    """Evaluate multiclass model. Returns macro-F1, top-3, top-5 accuracy."""
    y_prob = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1).astype(np.int64)
    y_true = y_test.astype(np.int64)
    return {
        "accuracy":     float(accuracy_score(y_true, y_pred)),
        "macro_f1":     float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "top3_accuracy": top_k_accuracy(y_true, y_prob, 3),
        "top5_accuracy": top_k_accuracy(y_true, y_prob, 5),
    }

In [ ]:
# Curated experiment definitions.
# Each dict is consumed by the runner loop below.
EXPERIMENTS = [
    {
        "name": "food_containers",
        "type": "binary",
        "label_level": "coarse",
        "positive_names": ["food_containers"],
        "dropout": 0.3,
        "lr": 1e-4,
    },
    {
        "name": "aquatic_mammals",
        "type": "binary",
        "label_level": "coarse",
        "positive_names": ["aquatic_mammals"],
        "dropout": 0.3,
        "lr": 3e-5,
    },
    {
        "name": "skyscraper",
        "type": "binary",
        "label_level": "fine",
        "positive_names": ["skyscraper"],
        "dropout": 0.3,
        "lr": 1e-5,
    },
    {
        "name": "orange",
        "type": "binary",
        "label_level": "fine",
        "positive_names": ["orange"],
        "dropout": 0.3,
        "lr": 1e-5,
    },
    {
        "name": "orange_aug_light",
        "type": "binary",
        "label_level": "fine",
        "positive_names": ["orange"],
        "dropout": 0.3,
        "lr": 1e-5,
        "augmentation": True,   # attach make_augmentation_layer()
    },
    {
        "name": "coarse_multiclass",
        "type": "multiclass",
        "dropout": 0.3,
        "lr": 3e-5,
    },
]

# To add another task, append a dict following the same schema above.
# Example (commented out):
# {"name": "cattle", "type": "binary", "label_level": "fine",
#  "positive_names": ["cattle"], "dropout": 0.3, "lr": 1e-4}

In [ ]:
all_results = []

for exp in EXPERIMENTS:
    name = exp["name"]
    epochs = (FAST_EPOCHS if FAST_DEV_RUN else DEFAULT_EPOCHS)[name]
    subset = FAST_SUBSET if FAST_DEV_RUN else None
    lr = exp["lr"]
    dropout = exp["dropout"]
    is_multi = exp["type"] == "multiclass"

    print(f"\n{'='*60}")
    print(f"Experiment: {name}  |  epochs={epochs}  |  lr={lr}")
    print(f"{'='*60}")

    # --- Build task labels ---
    if is_multi:
        y_tr_full, y_te_full, desc = multiclass_task()
        num_classes = len(np.unique(y_tr_full))
    else:
        y_tr_full, y_te_full, desc = binary_task(
            exp["label_level"], exp["positive_names"]
        )
        num_classes = 1

    # --- Subset (FAST_DEV_RUN) ---
    img_tr, y_tr_s = stratified_subset(
        train_images, y_tr_full, subset_size=subset, seed=SEED
    )
    img_te, y_te_s = stratified_subset(
        test_images, y_te_full, subset_size=subset, seed=SEED + 1
    )

    # --- Train/val split ---
    x_tr, y_tr, x_val, y_val = stratified_train_val_split(
        img_tr, y_tr_s, val_fraction=0.1, seed=SEED
    )

    # --- Pipelines ---
    train_ds = make_pipeline(x_tr, y_tr, batch_size=64, shuffle=True, seed=SEED)
    val_ds   = make_pipeline(x_val, y_val, batch_size=64, shuffle=False)
    test_ds  = make_pipeline(img_te, y_te_s, batch_size=64, shuffle=False)

    # --- Class weights (binary only) ---
    if num_classes == 1:
        cw = compute_balanced_class_weights(y_tr)
    else:
        cw = None

    # --- Model ---
    aug_layer = make_augmentation_layer() if exp.get("augmentation") else None
    tf.keras.backend.clear_session()
    model = build_baseline_cnn(dropout=dropout, num_classes=num_classes,
                               augmentation=aug_layer)

    loss = (tf.keras.losses.BinaryCrossentropy(from_logits=False)
            if num_classes == 1
            else tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=loss,
        metrics=["accuracy"],
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True
        )
    ]
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=epochs, class_weight=cw, callbacks=callbacks, verbose=2,
    )

    # --- Evaluate ---
    if num_classes == 1:
        metrics = evaluate_binary(model, val_ds, test_ds, y_val, y_te_s)
    else:
        metrics = evaluate_multiclass(model, test_ds, y_te_s)

    metrics["name"] = name
    metrics["desc"] = desc
    all_results.append(metrics)
    print(f"  result: {metrics}")

In [ ]:
print("\n" + "=" * 90)
print("RESULT TABLE")
print("=" * 90)

# Binary rows
print(f"\n{'Task':<22} {'thresh':>6} {'P':>6} {'R':>6} {'F1':>6} {'ROC-AUC':>8} {'PR-AUC':>7}")
print("-" * 65)
for r in all_results:
    if "f1" in r:
        print(
            f"{r['name']:<22} {r['threshold']:>6.2f} "
            f"{r['precision']:>6.3f} {r['recall']:>6.3f} {r['f1']:>6.3f} "
            f"{r['roc_auc']:>8.3f} {r['pr_auc']:>7.3f}"
        )

# Multiclass rows
print(f"\n{'Task':<22} {'acc':>6} {'macro-F1':>9} {'top-3':>6} {'top-5':>6}")
print("-" * 55)
for r in all_results:
    if "macro_f1" in r:
        print(
            f"{r['name']:<22} {r['accuracy']:>6.3f} "
            f"{r['macro_f1']:>9.3f} {r['top3_accuracy']:>6.3f} {r['top5_accuracy']:>6.3f}"
        )
print("=" * 90)

## Interpretation

**Binary tasks:** `skyscraper` typically achieves the highest F1/PR-AUC among fine tasks
at `lr=1e-5` with long training; `food_containers` is the strongest coarse baseline.
`orange_aug_light` trades precision for recall vs. plain `orange` — it reduces false
negatives but adds false positives, so it is an augmentation **ablation** rather than
the selected best orange model. `aquatic_mammals` is the harder coarse task; the best
run is around `lr=3e-5`, `dropout=0.3`, 30 epochs.

**Multiclass:** The 20-way coarse baseline is intentionally modest. High top-5 accuracy
at modest macro-F1 shows the model ranks the true class plausibly but not distinctly —
motivating stronger architectures.

## Next Steps

The baseline CNN establishes lower-bound results per task. Planned improvements:

1. **Stronger CNN with BatchNorm + L2 regularization** — replace Dropout-only
   regularization with BatchNorm between conv layers and L2 weight decay.
2. **Transfer learning** — frozen EfficientNet/MobileNetV3/ResNet backbones for
   feature extraction, then partial fine-tuning. Expected large gain on fine tasks.
3. **Row-as-timestep sequential models** — RNN, LSTM, BiLSTM consuming each image
   as a T=32 sequence of 96-feature rows.
4. **Small ViT** — from-scratch patch-embedding attention model for architectural
   comparison on the same binary tasks.